In [1]:
import importlib
import pandas as pd
from IPython.display import display
import framework
importlib.reload(framework)
from framework import WheatFramework, generate_group_folds,save_model, save_results, run_fold_with_trainer
import openpyxl

import importlib
import framework
importlib.reload(framework)
from framework import WheatFramework, generate_group_folds, save_model, save_results, run_fold_with_trainer

import os
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


Device: cuda


CHANGE ACCORDING TO YOUR PATHS !!

In [2]:

# Update paths if different on your machine
EXCEL_PATH = r".\labeling.xlsx"
DATA_PATH  = r".\data"
RESULTS_DIR = "results_rnn"

os.makedirs(RESULTS_DIR, exist_ok=True)


In [3]:

wf = WheatFramework(excel_path=EXCEL_PATH, root_dir=DATA_PATH)

try:
    meta_df = wf.load_and_preprocess()
except Exception as e:
    print("Error loading data:", e)
    meta_df = pd.DataFrame()

if not meta_df.empty:
    X, y, folds = wf.get_splits(meta_df, n_splits=5)
    print(f"Total matched images: {len(X)}")

    # Print label mapping (encoded -> string label)
    classes = list(wf.le.classes_)
    mapping = {i: cls for i, cls in enumerate(classes)}
    print("\nLabel mapping (encoded -> label):")
    for k, v in mapping.items():
        print(k, ":", v)

    print("\nSample matched paths and labels:")
    display(meta_df[['path', 'label']].head(8))

    # Visual verification: display one random sample image and its label
    wf.get_sample(X, y)
else:
    print("No images matched. Review previous prints above for column mapping and search path.")


Error loading data: 'WheatFramework' object has no attribute 'load_and_preprocess'
No images matched. Review previous prints above for column mapping and search path.


In [4]:
# Count images by group and stage
import pandas as pd

if 'meta_df' not in globals() or meta_df.empty:
    print("No `meta_df` available. Run the loader cell first (the one that creates `meta_df`).")
else:
    ct = pd.crosstab(meta_df['group_id'], meta_df['label'])
    ct['total'] = ct.sum(axis=1)
    ct = ct.sort_values('total', ascending=False)

    print('\nImages per group × stage (top 20 groups):')
    display(ct.head(20))

    print('\nStage totals:')
    stage_totals = ct.drop(columns=['total']).sum(axis=0).to_frame(name='count')
    display(stage_totals)


No `meta_df` available. Run the loader cell first (the one that creates `meta_df`).


In [5]:
if 'meta_df' in globals() and not meta_df.empty:
    example_folds = generate_group_folds(meta_df, n_train=8, n_test=2, num_folds=5, random_state=42)
    print('\nGenerated', len(example_folds), 'folds. Sample sizes:')
    for i, (tr, te) in enumerate(example_folds, 1):
        print(f' Fold {i}: train_samples={len(tr)}, test_samples={len(te)}')
else:
    print('meta_df not found — run loader cell first to generate folds.')


meta_df not found — run loader cell first to generate folds.


YOUR CODE HERE

In [6]:

# ---------------------
# EXAMPLES / TEMPLATES
# ---------------------

# Example: simple sklearn-like trainer function (for models with fit/predict APIs)
# def sklearn_trainer(train_df, test_df, fold_id):
#     # transform paths -> features as needed (e.g., load images and compute embeddings)
#     X_train, y_train = preprocess_paths_to_array(train_df['path'], train_df['label'])
#     X_test, y_test = preprocess_paths_to_array(test_df['path'], test_df['label'])
#     from sklearn.ensemble import RandomForestClassifier
#     model = RandomForestClassifier(n_estimators=100)
#     model.fit(X_train, y_train)
#     preds = model.predict(X_test)
#     # build predictions DataFrame
#     pred_df = pd.DataFrame({'path': test_df['path'], 'true': y_test, 'pred': preds})
#     metrics = {'accuracy': (pred_df['true'] == pred_df['pred']).mean()}
#     return {'model': model, 'predictions': pred_df, 'metrics': metrics}

# Example: PyTorch training skeleton (user must implement dataset, dataloader, model, optimizer)
# def pytorch_trainer(train_df, test_df, fold_id):
#     import torch
#     # implement Dataset that loads images from `path` and returns tensors + labels
#     train_dataset = MyImageDataset(train_df)
#     test_dataset = MyImageDataset(test_df)
#     train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
#     test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32)
#     model = MyNet()
#     optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
#     loss_fn = torch.nn.CrossEntropyLoss()
#     # training loop ...
#     # after training produce preds and metrics
#     pred_df = pd.DataFrame({'path': test_df['path'], 'true': true_labels, 'pred': pred_labels})
#     metrics = {'accuracy': accuracy}
#     return {'model': model, 'predictions': pred_df, 'metrics': metrics}

# Example: YOLO/Ultralytics style (pseudo)
# def yolov5_trainer(train_df, test_df, fold_id):
#     # prepare yaml/train.txt and yaml/val.txt listing image paths
#     # call ultralytics/train.py with appropriate args
#     # after training, load results and return predictions+metrics
#     pass

# ---------------------
# Quick sanity check: create a small set of folds and print counts


In [8]:
wf = WheatFramework(excel_path=EXCEL_PATH, root_dir=DATA_PATH)
meta_df = wf.get_dataframe()

if meta_df.empty:
    raise RuntimeError("Hiç görüntü eşleşmedi. Excel/dizin yollarını ve tarih formatını kontrol edin.")

print(f"\nToplam eşleşen görüntü: {len(meta_df)}")
print(f"Grup sayısı (station-year): {meta_df['group_id'].nunique()}")
print(f"Etiket dağılımı:\n{meta_df['label'].value_counts().to_string()}")
meta_df.head()

Searching for images in: c:\Users\PC\Desktop\AI_TARBIL\code\data
Found image: c:\Users\PC\Desktop\AI_TARBIL\code\data\02.02\2014\K1\1X\02_02-2014_01_01-10_10-K1-1X.jpeg (matched date in filename) -   Parsed date 1: 2014-01-01
Found image: c:\Users\PC\Desktop\AI_TARBIL\code\data\02.02\2014\K1\1X\02_02-2014_01_02-10_10-K1-1X.jpeg (matched date in filename) -   Parsed date 1: 2014-01-02
Found image: c:\Users\PC\Desktop\AI_TARBIL\code\data\02.02\2014\K1\1X\02_02-2014_01_03-10_11-K1-1X.jpeg (matched date in filename) -   Parsed date 1: 2014-01-03
Found image: c:\Users\PC\Desktop\AI_TARBIL\code\data\02.02\2014\K1\1X\02_02-2014_01_05-10_11-K1-1X.jpeg (matched date in filename) -   Parsed date 1: 2014-01-05
Found image: c:\Users\PC\Desktop\AI_TARBIL\code\data\02.02\2014\K1\1X\02_02-2014_01_06-10_12-K1-1X.jpeg (matched date in filename) -   Parsed date 1: 2014-01-06
Found image: c:\Users\PC\Desktop\AI_TARBIL\code\data\02.02\2014\K1\1X\02_02-2014_01_07-09_53-K1-1X.jpeg (matched date in filename)

,path,label,group_id,station_year
0,c:\Users\PC\Desktop\AI_TARBIL\code\data\02.02\...,PS0,1,02.02_2014
1,c:\Users\PC\Desktop\AI_TARBIL\code\data\02.02\...,PS0,1,02.02_2014
2,c:\Users\PC\Desktop\AI_TARBIL\code\data\02.02\...,PS0,1,02.02_2014
3,c:\Users\PC\Desktop\AI_TARBIL\code\data\02.02\...,PS0,1,02.02_2014
4,c:\Users\PC\Desktop\AI_TARBIL\code\data\02.02\...,PS0,1,02.02_2014


---
## ── COLAB GPU BLOKLARI ──
> Aşağıdaki hücreleri sırayla çalıştır. Cell 0-9 orjinal, dokunulmadı.

In [ ]:
# # ── BLOK 1: Google Drive Bağla ───────────────────────────────────────────────
# from google.colab import drive
# drive.mount("/content/drive")
# print("Drive bağlandı.")

ModuleNotFoundError: No module named 'google.colab'

In [11]:
# ── BLOK 2: Yollar (Drive içindeki klasöre göre düzenle) ─────────────────────
import os, re, math, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

# ⚠️  Kendi drive yoluna göre sadece bu satırı güncelle:
BASE = "./"

EXCEL_PATH  = os.path.join(BASE, "labeling.xlsx")
DATA_PATH   = os.path.join(BASE, "data")
RESULTS_DIR = "/content/results_colab"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Framework dosyasını Drive'dan kopyala (import için gerekli)
import shutil
shutil.copy(os.path.join(BASE, "framework.py"), "/content/framework.py")

import importlib, sys
sys.path.insert(0, "/content")
import framework; importlib.reload(framework)
from framework import WheatFramework, generate_group_folds, save_model, save_results, run_fold_with_trainer

print("Yollar:", EXCEL_PATH)
print("Data :", DATA_PATH)

Yollar: ./labeling.xlsx
Data : ./data


In [12]:
# ── BLOK 3: GPU & Kütüphaneler ───────────────────────────────────────────────
import torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as tvm
from torch.cuda.amp import GradScaler, autocast
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU  : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    USE_AMP = True   # Mixed precision — ücretsiz hız
else:
    print("  ⚠️  GPU yok. Colab → Runtime > Change runtime type > T4 GPU seç")
    USE_AMP = False

Device : cuda
  GPU  : NVIDIA GeForce RTX 4090
  VRAM : 25.8 GB


In [14]:
# ── BLOK 4: Veri Yükle + Tarih Çıkar ────────────────────────────────────────
wf = WheatFramework(excel_path=EXCEL_PATH, root_dir=DATA_PATH)
meta_df = wf.get_dataframe()
if meta_df.empty:
    raise RuntimeError("Görüntü eşleşmedi. Yolları kontrol et.")

_DATE_RE = re.compile(r"(\d{4})[_-](\d{2})[_-](\d{2})|(\d{4})(\d{2})(\d{2})")
def extract_date(path):
    m = _DATE_RE.search(os.path.basename(path))
    if not m: return pd.NaT
    g = m.groups()
    y,mo,d = (g[0],g[1],g[2]) if g[0] else (g[3],g[4],g[5])
    try: return pd.Timestamp(f"{y}-{mo}-{d}")
    except: return pd.NaT

meta_df["img_date"] = meta_df["path"].apply(extract_date)
meta_df = meta_df.dropna(subset=["img_date"])
meta_df = meta_df.sort_values(["group_id","img_date"]).reset_index(drop=True)

LABEL_NAMES = sorted(meta_df["label"].unique())
NUM_CLASSES  = len(LABEL_NAMES)
LABEL2IDX    = {l:i for i,l in enumerate(LABEL_NAMES)}
IDX2LABEL    = {i:l for l,i in LABEL2IDX.items()}

print(f"Toplam görüntü : {len(meta_df)}")
print(f"Sınıflar       : {LABEL_NAMES}")
print(meta_df["label"].value_counts().sort_index().to_string())

Searching for images in: c:\Users\PC\Desktop\AI_TARBIL\code\data
Found image: c:\Users\PC\Desktop\AI_TARBIL\code\data\02.02\2014\K1\1X\02_02-2014_01_01-10_10-K1-1X.jpeg (matched date in filename) -   Parsed date 1: 2014-01-01
Found image: c:\Users\PC\Desktop\AI_TARBIL\code\data\02.02\2014\K1\1X\02_02-2014_01_02-10_10-K1-1X.jpeg (matched date in filename) -   Parsed date 1: 2014-01-02
Found image: c:\Users\PC\Desktop\AI_TARBIL\code\data\02.02\2014\K1\1X\02_02-2014_01_03-10_11-K1-1X.jpeg (matched date in filename) -   Parsed date 1: 2014-01-03
Found image: c:\Users\PC\Desktop\AI_TARBIL\code\data\02.02\2014\K1\1X\02_02-2014_01_05-10_11-K1-1X.jpeg (matched date in filename) -   Parsed date 1: 2014-01-05
Found image: c:\Users\PC\Desktop\AI_TARBIL\code\data\02.02\2014\K1\1X\02_02-2014_01_06-10_12-K1-1X.jpeg (matched date in filename) -   Parsed date 1: 2014-01-06
Found image: c:\Users\PC\Desktop\AI_TARBIL\code\data\02.02\2014\K1\1X\02_02-2014_01_07-09_53-K1-1X.jpeg (matched date in filename)

In [15]:
# ── BLOK 5: Hiperparametreler (Colab T4 için optimize) ───────────────────────
CFG = {
    "img_size"      : 224,
    "img_mean"      : [0.485, 0.456, 0.406],
    "img_std"       : [0.229, 0.224, 0.225],

    # Backbone seçenekleri:
    #   "efficientnet_b2"  → en iyi doğruluk, T4'te iyi hız
    #   "efficientnet_b0"  → iyi denge
    #   "resnet34"         → en hızlı
    "backbone"      : "efficientnet_b2",
    "embed_dim"     : 1408,   # b0=1280, b2=1408, resnet34=512

    "seq_len"       : 24,     # daha uzun temporal context
    "stride"        : 4,

    "hidden_dim"    : 384,    # GPU var, daha geniş LSTM
    "num_layers"    : 2,
    "dropout"       : 0.4,
    "bidirectional" : True,

    # Phase 1 — CNN frozen
    "p1_epochs"     : 12,
    "p1_lr"         : 3e-3,

    # Phase 2 — CNN unfrozen
    "p2_epochs"     : 15,
    "p2_lr_lstm"    : 3e-4,
    "p2_lr_cnn"     : 1e-5,

    "batch_size"    : 24,     # T4 VRAM'a göre artır (OOM olursa azalt)
    "weight_decay"  : 1e-4,
    "grad_clip"     : 1.0,
    "patience"      : 7,
    "focal_gamma"   : 2.0,
    "label_smooth"  : 0.05,   # label smoothing — overfit azaltır

    # TTA (Test Time Augmentation) — inference'da kaç augmented pass
    "tta_n"         : 5,

    "n_train"       : 8,
    "n_test"        : 2,
    "num_folds"     : 5,
    "random_state"  : 42,
}

_DIM = {"resnet18":512,"resnet34":512,"resnet50":2048,
        "efficientnet_b0":1280,"efficientnet_b2":1408,"efficientnet_b4":1792}
CFG["embed_dim"] = _DIM.get(CFG["backbone"], CFG["embed_dim"])

print(f"Backbone  : {CFG['backbone']}  embed_dim={CFG['embed_dim']}")
print(f"Phase 1   : {CFG['p1_epochs']}ep  |  Phase 2: {CFG['p2_epochs']}ep")
print(f"Batch     : {CFG['batch_size']}  |  AMP: {USE_AMP}")

Backbone  : efficientnet_b2  embed_dim=1408
Phase 1   : 12ep  |  Phase 2: 15ep
Batch     : 24  |  AMP: True


In [16]:
# ── BLOK 6: Dataset ───────────────────────────────────────────────────────────
class PhenologyDataset(Dataset):
    """
    mode="all"  → her frame için ayrı etiket  (daha fazla gradient)
    mode="last" → sadece son frame  (TTA için)
    """
    def __init__(self, df, label2idx, seq_len=24, stride=4,
                 transform=None, mode="all"):
        self.df = df.reset_index(drop=True)
        self.l2i = label2idx
        self.seq_len = seq_len; self.stride = stride
        self.transform = transform; self.mode = mode
        self.seqs = self._build()

    def _build(self):
        seqs = []
        for _, grp in self.df.groupby("group_id", sort=False):
            grp = grp.sort_values("img_date").reset_index(drop=True)
            n = len(grp)
            if n < self.seq_len:
                seqs.append((grp, [0]*(self.seq_len-n)+list(range(n))))
            else:
                for s in range(0, n-self.seq_len+1, self.stride):
                    seqs.append((grp, list(range(s, s+self.seq_len))))
        return seqs

    def __len__(self): return len(self.seqs)

    def _load(self, path):
        try:   img = Image.open(path).convert("RGB")
        except: img = Image.fromarray(np.zeros((224,224,3), dtype=np.uint8))
        return self.transform(img) if self.transform else T.ToTensor()(img)

    def __getitem__(self, idx):
        grp, fi = self.seqs[idx]
        frames = torch.stack([self._load(grp.iloc[i]["path"]) for i in fi])
        if self.mode == "last":
            return frames, torch.tensor(self.l2i[grp.iloc[fi[-1]]["label"]], dtype=torch.long)
        return frames, torch.tensor(
            [self.l2i[grp.iloc[i]["label"]] for i in fi], dtype=torch.long)


# Dönüşümler — TTA için ayrı strong augment
TRAIN_TF = T.Compose([
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(p=0.2),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.RandomGrayscale(p=0.05),
    T.RandomErasing(p=0.1, scale=(0.02,0.1)),   # rastgele kısım sil
    T.ToTensor(),
    T.Normalize(CFG["img_mean"], CFG["img_std"]),
])
VAL_TF = T.Compose([
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.ToTensor(), T.Normalize(CFG["img_mean"], CFG["img_std"]),
])
TTA_TF = T.Compose([   # TTA passları için hafif augment
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.RandomHorizontalFlip(), T.RandomRotation(5),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(), T.Normalize(CFG["img_mean"], CFG["img_std"]),
])
print("Dataset hazır.")

Dataset hazır.


In [17]:
# ── BLOK 7: Model — EfficientNet + BiLSTM + Temporal Attention ───────────────
class CNNEncoder(nn.Module):
    def __init__(self, backbone="efficientnet_b2", freeze=True):
        super().__init__()
        if "efficientnet" in backbone:
            w = "IMAGENET1K_V1"
            base = getattr(tvm, backbone)(weights=w)
            self.net = nn.Sequential(base.features, base.avgpool, nn.Flatten())
        else:
            base = getattr(tvm, backbone)(weights="IMAGENET1K_V1")
            self.net = nn.Sequential(*list(base.children())[:-1], nn.Flatten())
        if freeze:
            for p in self.net.parameters(): p.requires_grad = False

    def unfreeze(self, layers_from_end=None):
        """
        layers_from_end=None → hepsini aç
        layers_from_end=3    → sadece son 3 blok (daha dikkatli fine-tune)
        """
        children = list(self.net.children())
        to_unfreeze = children if layers_from_end is None else children[-layers_from_end:]
        for block in to_unfreeze:
            for p in block.parameters(): p.requires_grad = True

    def forward(self, x): return self.net(x)


class TemporalAttention(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.fc = nn.Linear(h, 1)
    def forward(self, x):          # (B,T,H)
        w = F.softmax(self.fc(x).squeeze(-1), dim=-1).unsqueeze(-1)
        return (x*w).sum(1), w.squeeze(-1)


class PhenologyRNN(nn.Module):
    def __init__(self, num_classes, cfg, mode="all"):
        super().__init__()
        self.mode = mode
        self.encoder = CNNEncoder(cfg["backbone"], freeze=True)
        dirs = 2 if cfg["bidirectional"] else 1
        self.lstm = nn.LSTM(
            input_size=cfg["embed_dim"], hidden_size=cfg["hidden_dim"],
            num_layers=cfg["num_layers"], batch_first=True,
            dropout=cfg["dropout"] if cfg["num_layers"]>1 else 0,
            bidirectional=cfg["bidirectional"],
        )
        H = cfg["hidden_dim"] * dirs
        self.attn = TemporalAttention(H)
        self.head = nn.Sequential(
            nn.LayerNorm(H), nn.Dropout(cfg["dropout"]), nn.Linear(H, num_classes)
        )

    def forward(self, x):
        B,T,C,H,W = x.shape
        emb = self.encoder(x.view(B*T,C,H,W)).view(B,T,-1)
        out,_ = self.lstm(emb)
        if self.mode == "all": return self.head(out)   # (B,T,C)
        ctx, attn_w = self.attn(out)
        return self.head(ctx), attn_w                  # (B,C), (B,T)


# Test
with torch.no_grad():
    _m = PhenologyRNN(NUM_CLASSES, CFG, "all").to(DEVICE)
    _x = torch.randn(2, CFG["seq_len"], 3, 224, 224).to(DEVICE)
    tr = sum(p.numel() for p in _m.parameters() if p.requires_grad)
    tt = sum(p.numel() for p in _m.parameters())
    print(f"Çıktı : {list(_m(_x).shape)}")
    print(f"Param : {tr:,} eğitilebilir / {tt:,} toplam")
    del _m, _x

Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to C:\Users\PC/.cache\torch\hub\checkpoints\efficientnet_b2_rwightman-c35c1473.pth


100%|██████████| 35.2M/35.2M [00:02<00:00, 17.7MB/s]


Çıktı : [2, 24, 8]
Param : 9,064,713 eğitilebilir / 16,765,707 toplam


In [18]:
# ── BLOK 8: Focal Loss + Label Smoothing + Sınıf Ağırlıkları ─────────────────
class FocalLoss(nn.Module):
    """Focal + Label Smoothing birleşimi."""
    def __init__(self, gamma=2.0, weight=None, smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.smoothing = smoothing

    def forward(self, logits, targets):
        # Label smoothing CE
        n_cls = logits.size(-1)
        log_p = F.log_softmax(logits, dim=-1)
        # one-hot smooth target
        with torch.no_grad():
            smooth_t = torch.full_like(log_p, self.smoothing/(n_cls-1))
            smooth_t.scatter_(-1, targets.unsqueeze(-1), 1-self.smoothing)
        ce = -(smooth_t * log_p).sum(-1)
        if self.weight is not None:
            ce = ce * self.weight[targets]
        pt = torch.exp(-ce)
        return ((1-pt)**self.gamma * ce).mean()


def get_class_weights(df, label2idx, device):
    counts = df["label"].value_counts()
    n, k = len(df), len(label2idx)
    w = np.array([n/(k*counts.get(l,1)) for l in sorted(label2idx, key=label2idx.get)])
    return torch.tensor(w/w.sum()*k, dtype=torch.float32, device=device)

print("Focal + LabelSmooth Loss hazır.")

Focal + LabelSmooth Loss hazır.


In [19]:
# ── BLOK 9: Eğitim Döngüsü (AMP destekli) ────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion, device, scaler, grad_clip=1.0):
    model.train()
    ls, correct, total = 0., 0, 0
    for frames, labels in loader:
        frames = frames.to(device, non_blocking=True)
        optimizer.zero_grad()
        with autocast(enabled=USE_AMP):
            out = model(frames)
            if out.dim() == 3:          # mode="all"
                B,T,C = out.shape
                loss  = criterion(out.view(B*T,C), labels.to(device).view(B*T))
                preds = out.argmax(-1).view(-1); tgts = labels.to(device).view(-1)
            else:
                loss  = criterion(out, labels.to(device))
                preds = out.argmax(-1);   tgts = labels.to(device)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer); scaler.update()
        ls += loss.item()*tgts.size(0)
        correct += (preds==tgts).sum().item(); total += tgts.size(0)
    return ls/total, correct/total


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    ls, correct, total = 0., 0, 0
    all_p, all_t = [], []
    for frames, labels in loader:
        frames = frames.to(device, non_blocking=True)
        with autocast(enabled=USE_AMP):
            out = model(frames)
            logits = out[0] if isinstance(out, tuple) else out
            if logits.dim() == 3:
                B,T,C = logits.shape
                loss  = criterion(logits.view(B*T,C), labels.to(device).view(B*T))
                preds = logits.argmax(-1).view(-1); tgts = labels.to(device).view(-1)
            else:
                loss  = criterion(logits, labels.to(device))
                preds = logits.argmax(-1); tgts = labels.to(device)
        ls += loss.item()*tgts.size(0)
        correct += (preds==tgts).sum().item(); total += tgts.size(0)
        all_p.extend(preds.cpu().tolist()); all_t.extend(tgts.cpu().tolist())
    return ls/total, correct/total, all_p, all_t


@torch.no_grad()
def tta_predict(model, df, label2idx, seq_len, stride, device, n=5):
    """
    Test Time Augmentation: n farklı augmented pass'ı average'la.
    Tek başına ~2-4% accuracy kazandırır.
    """
    model.eval(); model.mode = "attn"
    all_logits_list = []
    for _ in range(n):
        ds = PhenologyDataset(df, label2idx, seq_len, stride, TTA_TF, mode="last")
        ld = DataLoader(ds, 16, shuffle=False, num_workers=2, pin_memory=True)
        fold_logits = []
        for frames, _ in ld:
            out = model(frames.to(device))
            logits = out[0] if isinstance(out, tuple) else out
            fold_logits.append(logits.cpu())
        all_logits_list.append(torch.cat(fold_logits, 0))  # (N, C)
    avg = torch.stack(all_logits_list).mean(0)             # (N, C)
    return avg.argmax(-1).tolist()


def cosine_warmup(optimizer, warmup, total):
    def f(ep):
        if ep < warmup: return (ep+1)/warmup
        p = (ep-warmup)/max(1, total-warmup)
        return 0.5*(1+math.cos(math.pi*p))
    return optim.lr_scheduler.LambdaLR(optimizer, f)

print("Eğitim fonksiyonları (AMP + TTA) hazır.")

Eğitim fonksiyonları (AMP + TTA) hazır.


In [20]:
# ── BLOK 10: rnn_trainer_colab — İki Fazlı + AMP + TTA ──────────────────────
def rnn_trainer_colab(train_df, test_df, fold_id):
    print(f"\n{'='*65}")
    print(f"  FOLD {fold_id}   train={len(train_df)}   test={len(test_df)}")
    print(f"{'='*65}")

    nw = 2   # Colab Linux'ta num_workers>0 güvenli
    train_ds = PhenologyDataset(train_df, LABEL2IDX, CFG["seq_len"], CFG["stride"], TRAIN_TF, "all")
    test_ds  = PhenologyDataset(test_df,  LABEL2IDX, CFG["seq_len"], CFG["stride"], VAL_TF,   "all")
    print(f"  Sequences → train: {len(train_ds)}  test: {len(test_ds)}")

    train_ld = DataLoader(train_ds, CFG["batch_size"], shuffle=True,  num_workers=nw, pin_memory=True)
    test_ld  = DataLoader(test_ds,  CFG["batch_size"], shuffle=False, num_workers=nw, pin_memory=True)

    cw = get_class_weights(train_df, LABEL2IDX, DEVICE)
    crit = FocalLoss(CFG["focal_gamma"], cw, CFG["label_smooth"])
    scaler = GradScaler(enabled=USE_AMP)
    best_state, best_acc, history = None, -1., []

    # ── Phase 1: CNN Frozen ───────────────────────────────────────────────────
    print("\n🔒 Phase 1: CNN frozen")
    model = PhenologyRNN(NUM_CLASSES, CFG, "all").to(DEVICE)
    opt1  = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=CFG["p1_lr"], weight_decay=CFG["weight_decay"])
    sch1  = cosine_warmup(opt1, 2, CFG["p1_epochs"])
    pc = 0
    for ep in range(1, CFG["p1_epochs"]+1):
        tl,ta = train_epoch(model, train_ld, opt1, crit, DEVICE, scaler, CFG["grad_clip"])
        vl,va,_,_ = eval_epoch(model, test_ld, crit, DEVICE)
        sch1.step()
        history.append({"phase":1,"ep":ep,"tl":tl,"ta":ta,"vl":vl,"va":va})
        print(f"  P1 {ep:02d}/{CFG['p1_epochs']}  tr={ta:.3f}  va={va:.3f}  lr={opt1.param_groups[0]['lr']:.2e}")
        if va > best_acc:
            best_acc,pc = va,0
            best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            pc+=1
            if pc>=CFG["patience"]: print(f"  [early stop P1@{ep}]"); break

    # ── Phase 2: Son 4 blok unfreeze ─────────────────────────────────────────
    print("\n🔓 Phase 2: Son bloklar unfreeze → fine-tune")
    model.load_state_dict(best_state)
    model.encoder.unfreeze(layers_from_end=4)
    opt2 = optim.AdamW([
        {"params": model.encoder.parameters(), "lr": CFG["p2_lr_cnn"]},
        {"params": model.lstm.parameters(),    "lr": CFG["p2_lr_lstm"]},
        {"params": model.attn.parameters(),    "lr": CFG["p2_lr_lstm"]},
        {"params": model.head.parameters(),    "lr": CFG["p2_lr_lstm"]},
    ], weight_decay=CFG["weight_decay"])
    sch2 = cosine_warmup(opt2, 1, CFG["p2_epochs"])
    pc = 0
    for ep in range(1, CFG["p2_epochs"]+1):
        tl,ta = train_epoch(model, train_ld, opt2, crit, DEVICE, scaler, CFG["grad_clip"])
        vl,va,_,_ = eval_epoch(model, test_ld, crit, DEVICE)
        sch2.step()
        history.append({"phase":2,"ep":ep,"tl":tl,"ta":ta,"vl":vl,"va":va})
        print(f"  P2 {ep:02d}/{CFG['p2_epochs']}  tr={ta:.3f}  va={va:.3f}  cnn_lr={opt2.param_groups[0]['lr']:.1e}")
        if va > best_acc:
            best_acc,pc = va,0
            best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            pc+=1
            if pc>=CFG["patience"]: print(f"  [early stop P2@{ep}]"); break

    # ── Final: Normal eval ───────────────────────────────────────────────────
    model.load_state_dict(best_state)
    model.mode = "all"
    _,_,preds_idx,true_idx = eval_epoch(model, test_ld, crit, DEVICE)
    acc_plain = accuracy_score(true_idx, preds_idx)

    # ── TTA eval ─────────────────────────────────────────────────────────────
    print(f"\n  🔁 TTA ({CFG['tta_n']}x) hesaplanıyor...")
    tta_preds = tta_predict(model, test_df, LABEL2IDX,
                            CFG["seq_len"], CFG["stride"], DEVICE, CFG["tta_n"])
    # TTA için true_idx: tek sequence başına tek etiket lazım
    ds_single = PhenologyDataset(test_df, LABEL2IDX, CFG["seq_len"], CFG["stride"], VAL_TF, "last")
    tta_true  = [ds_single[i][1].item() for i in range(len(ds_single))]
    acc_tta   = accuracy_score(tta_true, tta_preds)
    wf1_tta   = f1_score(tta_true, tta_preds, average="weighted", zero_division=0)
    mf1_tta   = f1_score(tta_true, tta_preds, average="macro",    zero_division=0)
    pcf1_tta  = f1_score(tta_true, tta_preds, average=None,       zero_division=0)

    print(f"  Plain acc : {acc_plain:.4f}")
    print(f"  TTA acc   : {acc_tta:.4f}  wF1={wf1_tta:.4f}  macroF1={mf1_tta:.4f}")

    metrics = {
        "fold": fold_id,
        "accuracy_plain": round(acc_plain, 4),
        "accuracy_tta":   round(acc_tta,   4),
        "weighted_f1":    round(float(wf1_tta), 4),
        "macro_f1":       round(float(mf1_tta), 4),
        "best_val_acc":   round(best_acc, 4),
        "per_class_f1":   {IDX2LABEL[i]: round(float(v),4) for i,v in enumerate(pcf1_tta)},
    }

    pred_df = pd.DataFrame({"true_idx": tta_true, "pred_idx": tta_preds,
                             "true_label":[IDX2LABEL[i] for i in tta_true],
                             "pred_label":[IDX2LABEL[i] for i in tta_preds]})
    fold_dir = os.path.join(RESULTS_DIR, f"fold_{fold_id}")
    os.makedirs(fold_dir, exist_ok=True)
    pd.DataFrame(history).to_csv(os.path.join(fold_dir,"history.csv"), index=False)
    model.mode = "all"

    return {"model": model, "predictions": pred_df, "metrics": metrics}

print("trainer_fn hazır.")

trainer_fn hazır.


In [ ]:
# ── BLOK 11: Fold'ları Çalıştır ──────────────────────────────────────────────
folds = generate_group_folds(
    meta_df, group_col="group_id",
    n_train=CFG["n_train"], n_test=CFG["n_test"],
    num_folds=CFG["num_folds"], random_state=CFG["random_state"],
)
print(f"Fold sayısı: {len(folds)}\n")

all_results = []
for fold_id, (train_idx, test_idx) in enumerate(folds, 1):
    res = run_fold_with_trainer(
        meta_df, train_idx=train_idx, test_idx=test_idx,
        trainer_fn=rnn_trainer_colab, out_dir=RESULTS_DIR,
        fold_id=fold_id, save_model_flag=True,
    )
    all_results.append(res)

Fold sayısı: 5


  FOLD 1   train=1361   test=381
  Sequences → train: 298  test: 85

🔒 Phase 1: CNN frozen


In [ ]:
# ── BLOK 12: Özet Tablo ──────────────────────────────────────────────────────
rows = []
for r in all_results:
    m = r["metrics"]
    rows.append({"fold": m["fold"], "acc_plain": m["accuracy_plain"],
                 "acc_tta": m["accuracy_tta"],
                 "weighted_f1": m["weighted_f1"], "macro_f1": m["macro_f1"]})
summary = pd.DataFrame(rows)
summary.loc["mean"] = summary.mean(numeric_only=True)
summary.loc["std"]  = summary.std(numeric_only=True)
print("📊 Fold Özeti")
print(summary.to_string())
summary.to_csv(os.path.join(RESULTS_DIR, "fold_summary.csv"))

In [ ]:
# ── BLOK 13: Görselleştirme ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = plt.cm.tab10.colors
for fi in range(len(all_results)):
    hist = pd.read_csv(os.path.join(RESULTS_DIR, f"fold_{fi+1}", "history.csv"))
    p1e = len(hist[hist["phase"]==1])
    ep  = list(range(len(hist)))
    axes[0].axvline(p1e, color=colors[fi], linestyle=":", alpha=0.4)
    axes[0].plot(ep, hist["tl"], c=colors[fi], label=f"F{fi+1} tr")
    axes[0].plot(ep, hist["vl"], c=colors[fi], linestyle="--")
    axes[1].plot(ep, hist["ta"], c=colors[fi])
    axes[1].plot(ep, hist["va"], c=colors[fi], linestyle="--")
for ax,t in zip(axes,["Loss","Accuracy"]):
    ax.set_title(t); ax.set_xlabel("Epoch"); ax.grid(alpha=0.3)
axes[0].legend(fontsize=7)
plt.suptitle("Eğitim Eğrileri  (noktalı çizgi = Phase 1→2)", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,"training_curves.png"), dpi=120); plt.show()

# Confusion Matrix
all_p, all_t = [], []
for fi in range(len(all_results)):
    pp = pd.read_csv(os.path.join(RESULTS_DIR, f"fold_{fi+1}", "predictions.csv"))
    all_p.extend(pp["pred_idx"]); all_t.extend(pp["true_idx"])

cm   = confusion_matrix(all_t, all_p)
cm_n = cm.astype(float)/cm.sum(axis=1,keepdims=True).clip(1)
fig,ax = plt.subplots(figsize=(9,7))
sns.heatmap(cm_n, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
ax.set_xlabel("Tahmin"); ax.set_ylabel("Gerçek")
ax.set_title("Normalize Confusion Matrix — Tüm Foldlar (TTA)")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,"confusion_matrix.png"), dpi=120); plt.show()

# Per-class F1
pcf1 = f1_score(all_t, all_p, average=None, zero_division=0)
print(classification_report(all_t, all_p, target_names=LABEL_NAMES, zero_division=0))
plt.figure(figsize=(8,4))
plt.bar(LABEL_NAMES, pcf1, color=["#e74c3c" if v<0.5 else "#27ae60" for v in pcf1])
plt.ylim(0,1); plt.axhline(0.5,color="gray",linestyle="--",alpha=0.5)
plt.title("Sınıf Bazlı F1  (kırmızı < 0.5)"); plt.ylabel("F1"); plt.grid(axis="y",alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,"per_class_f1.png"), dpi=120); plt.show()

In [ ]:
# ── BLOK 14: Attention Haritası ──────────────────────────────────────────────
def visualize_attention(model, df, seq_len, device, n=4):
    model.eval(); model.mode = "attn"
    ds = PhenologyDataset(df, LABEL2IDX, seq_len=seq_len, stride=seq_len,
                          transform=VAL_TF, mode="last")
    fig, axes = plt.subplots(n, 1, figsize=(14, 3*n))
    if n==1: axes=[axes]
    for ax, si in zip(axes, range(min(n, len(ds)))):
        frames, label = ds[si]
        with torch.no_grad():
            logits, w = model(frames.unsqueeze(0).to(device))
        pred = logits.argmax(-1).item()
        ax.bar(range(seq_len), w.squeeze().cpu().numpy(), color="steelblue")
        ax.set_title(f"True: {IDX2LABEL[label.item()]}  |  Pred: {IDX2LABEL[pred]}")
        ax.set_xlabel("Frame (zaman →)"); ax.set_ylabel("Attention")
    plt.suptitle("Temporal Attention Haritası", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR,"attention_map.png"), dpi=120); plt.show()
    model.mode = "all"

visualize_attention(all_results[-1]["model"],
                    meta_df.iloc[folds[-1][1]].reset_index(drop=True),
                    CFG["seq_len"], DEVICE)